In [1]:
import os
import cv2
import numpy as np
from tqdm import tqdm

dataset_path = r"C:\Users\chris\Documents\Research\dataset\OCTDL\data"
class_names = ["AMD", "DME", "ERM", "NO", "RAO", "RVO", "VID"]

target_size = 224

X = []
y = []

for class_name in class_names:
    
    class_folder = os.path.join(dataset_path, class_name)
    
    print("Processing class:", class_name)
    
    for image_name in tqdm(os.listdir(class_folder)):
        
        image_path = os.path.join(class_folder, image_name)
        
        image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        
        if image is None:
            print("Image not loaded:", image_path)
            continue
        
        # Apply CLAHE
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        image_clahe = clahe.apply(image)
        
        # Get height and width
        h, w = image_clahe.shape
        
        # Resize without distortion
        scale = target_size / max(h, w)
        
        new_h = int(h * scale)
        new_w = int(w * scale)
        
        resized = cv2.resize(image_clahe, (new_w, new_h), interpolation=cv2.INTER_AREA)
        
        # Create black 224x224 background
        padded = np.zeros((target_size, target_size), dtype=np.uint8)
        
        # Center padding
        top = (target_size - new_h) // 2
        left = (target_size - new_w) // 2
        
        padded[top:top + new_h, left:left + new_w] = resized
        
        # Normalize
        normalized = padded.astype("float32") / 255.0
        
        # Add channel dimension
        final_image = np.expand_dims(normalized, axis=0)
        
        # Store image and label
        X.append(final_image)
        y.append(class_name)

Processing class: AMD


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 885/885 [00:08<00:00, 104.99it/s]


Processing class: DME


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 143/143 [00:01<00:00, 125.77it/s]


Processing class: ERM


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 133/133 [00:01<00:00, 118.08it/s]


... (4 outputs left out)

Processing class: RVO



100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 93/93 [00:00<00:00, 117.35it/s]


Processing class: VID


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 58/58 [00:00<00:00, 122.67it/s]


In [2]:
X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1618, 1, 224, 224)
y shape: (1618,)


In [3]:
import pandas as pd

class_count = pd.Series(y).value_counts()
print(class_count)

AMD    885
NO     284
DME    143
ERM    133
RVO     93
VID     58
RAO     22
Name: count, dtype: int64


In [4]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Encoded labels shape:", y_encoded.shape)

print("Class mapping:")
for index, class_name in enumerate(label_encoder.classes_):
    print(index, ":", class_name)

Encoded labels shape: (1618,)
Class mapping:
0 : AMD
1 : DME
2 : ERM
3 : NO
4 : RAO
5 : RVO
6 : VID


In [5]:
np.save("X_preprocessed_conf1405.npy", X)
np.save("y_labels_conf1405.npy", y_encoded)

print("Preprocessed images and labels saved successfully.")

Preprocessed images and labels saved successfully.


In [2]:
X = np.load("X_preprocessed_conf1405.npy")
y_encoded = np.load("y_labels_conf1405.npy")